# 41 — Capabilities, one at a time (Group B)

Each capability added on the simplest tree that supports it, probit and logit as separate analyses. The logit 2PL/3PL cells estimate discrimination `a` and guessing `c` by MH (the package's logit-node capability).

## B1. Disengagement + 2PL node — discrimination `a`

## Setup — install the package, load the harness simulator

In [ ]:
import sys
!{sys.executable} -m pip install -e "C:\Users\manonfi\OneDrive - James Madison University\Spring 26 classes & projects\Practicum Python IRtree package\PACKAGE\Irtree"

In [ ]:
import sys, warnings, pathlib, importlib.util
import numpy as np, pandas as pd
np.set_printoptions(precision=4, suppress=True)
warnings.filterwarnings('ignore')

import irtree.tree_spec   as ts
import irtree.node_models as nm
import irtree.priors      as pr
import irtree.constraints as con
import irtree.ingest      as ing
import irtree.fit         as ft

# The data-generating code (simulate_irtree) lives in the test harness, NOT in
# the package: simulation is deliberately kept separate from fitting.  Load it
# the same way notebooks 40 & 50 do.
ROOT = pathlib.Path.cwd().parents[1]
_hspec = importlib.util.spec_from_file_location("_harness", ROOT / "tests" / "_harness.py")
h = importlib.util.module_from_spec(_hspec); _hspec.loader.exec_module(h)

# probit -> logit conversion, per parameter role (TESTING.md 9.3), for your
# own hand comparisons between the two fits.
D_PL = 1.702
def to_logit(value, role):
    if role == 'slope':        return D_PL * np.asarray(value)
    if role in ('location','correlation','difficulty'): return np.asarray(value)
    raise ValueError(role)

## Study size (you choose)

In [ ]:
# ===================== study size — set these yourself =====================
# One simulated dataset per scenario (a rehearsal / recovery study, not a tiered
# test). Evidence-mode warmup is the package default, so any fit with an MH block
# (logit nodes, 3PL guessing, an estimated correlation) needs N_ITER >= ~1600;
# pure-Gibbs probit Rasch has no such floor.
P, I     = 1500, 30
N_ITER   = 4000
N_CHAINS = 2
SEED     = 20250413
print(f'P={P}  I={I}  N_ITER={N_ITER}  N_CHAINS={N_CHAINS}  SEED={SEED}')

## Two small helpers for this notebook
`responses_frame` makes the simulated matrix look like a real response file for `ingest`; `compare` prints posterior vs truth. Everything else is called explicitly so you can rehearse the API.

In [ ]:
def responses_frame(Y):
    """Turn a simulated [P, I] response matrix into the kind of keyed table a
    real study would hand to ingest.load_data: one row per person, item ids as
    column headers, an explicit person_id. (load_data also reads a CSV path.)"""
    Y = np.asarray(Y); Pn, In = Y.shape
    df = pd.DataFrame(Y, columns=[f'item{i+1}' for i in range(In)])
    df.insert(0, 'person_id', [f'p{p+1}' for p in range(Pn)])
    return df

def person_frame(X, names):
    df = pd.DataFrame(np.asarray(X), columns=list(names))
    df.insert(0, 'person_id', [f'p{p+1}' for p in range(np.asarray(X).shape[0])])
    return df

def item_frame(Q, names):
    df = pd.DataFrame(np.asarray(Q), columns=list(names))
    df.insert(0, 'item_id', [f'item{i+1}' for i in range(np.asarray(Q).shape[0])])
    return df

def compare(true_vec, prefix, fit, label=''):
    """Posterior mean + 95% interval for every parameter named `prefix*`,
    next to the known truth. Rehearses pulling estimates out of fit.results.
    Returns the DataFrame so you can compare however you like."""
    names = fit.results[0].param_names
    cols  = [j for j, n in enumerate(names) if n.startswith(prefix)]
    draws = np.concatenate([np.asarray(r.chain, float) for r in fit.results])[:, cols]
    tbl = pd.DataFrame({
        'param'    : [names[j] for j in cols],
        'true'     : np.asarray(true_vec, float).ravel()[:len(cols)],
        'post_mean': draws.mean(0),
        'ci_2.5'   : np.percentile(draws, 2.5, axis=0),
        'ci_97.5'  : np.percentile(draws, 97.5, axis=0),
    })
    tbl['abs_err'] = (tbl['post_mean'] - tbl['true']).abs()
    if label: print(label)
    print(tbl.to_string(index=False,
          formatters={c: '{:+.3f}'.format for c in ['true','post_mean','ci_2.5','ci_97.5','abs_err']}))
    return tbl

**Simulate** (data-generating step — separate from the package)

In [ ]:
spec_dis = ts.disengagement_tree(binary=True)
rng = np.random.default_rng(SEED+10)
theta1, theta2 = rng.normal(0,1,P), rng.normal(0,1,P)
b1_true = rng.normal(0.6,0.7,I); b2_true = rng.normal(0,1,I)
a2_true = rng.uniform(0.7,1.6,I)                      # node-2 discrimination
truth = [dict(theta=theta1, b=b1_true), dict(theta=theta2, b=b2_true, a=a2_true)]
Y_probit = h.simulate_irtree(spec_dis, truth, P, I, np.random.default_rng(SEED+10), node_links=['probit','probit'])
Y_logit  = h.simulate_irtree(spec_dis, truth, P, I, np.random.default_rng(SEED+10), node_links=['logit','logit'])
resp_probit, resp_logit = responses_frame(Y_probit), responses_frame(Y_logit)

**Probit** — ingest, node models, fit, compare

In [ ]:
data = ing.load_data(resp_probit, spec=spec_dis, categories_start=1)
node_models = [nm.ProbitRaschNode(), nm.ProbitTwoPLNode()]   # default a prior -> TruncNormal Gibbs
fit = ft.irtree_fit(data, spec_dis, node_models, n_iter=N_ITER, n_chains=N_CHAINS, seed=SEED+10)
compare(a2_true, 'a_2_', fit, 'node-2 discrimination a (probit)')
compare(b2_true, 'b_2_', fit, 'node-2 difficulty b (probit)')

**Logit** — the same study, logit link

In [ ]:
data = ing.load_data(resp_logit, spec=spec_dis, categories_start=1)
node_models = [nm.LogitRaschNode(), nm.LogitTwoPLNode()]      # a estimated by MH (logit)
fit = ft.irtree_fit(data, spec_dis, node_models, n_iter=N_ITER, n_chains=N_CHAINS, seed=SEED+10)
print('a MH accept:', {k:round(v,3) for k,v in fit.results[0].accept_rates.items() if k.startswith('a_mh')})
compare(a2_true, 'a_2_', fit, 'node-2 discrimination a (logit, MH)')

## B2. Disengagement + 3PL node — discrimination `a` and guessing `c`

**Simulate** (data-generating step — separate from the package)

In [ ]:
spec_dis = ts.disengagement_tree(binary=True)
rng = np.random.default_rng(SEED+11)
theta1, theta2 = rng.normal(0,1,P), rng.normal(0,1,P)
b1_true = rng.normal(0.6,0.7,I); b2_true = rng.normal(0,1,I)
a2_true = rng.uniform(0.8,1.5,I); c2_true = np.full(I, 0.2)
truth = [dict(theta=theta1, b=b1_true), dict(theta=theta2, b=b2_true, a=a2_true, c=c2_true)]
Y_probit = h.simulate_irtree(spec_dis, truth, P, I, np.random.default_rng(SEED+11), node_links=['probit','probit'])
Y_logit  = h.simulate_irtree(spec_dis, truth, P, I, np.random.default_rng(SEED+11), node_links=['logit','logit'])
resp_probit, resp_logit = responses_frame(Y_probit), responses_frame(Y_logit)

**Probit** — ingest, node models, fit, compare

In [ ]:
data = ing.load_data(resp_probit, spec=spec_dis, categories_start=1)
node_models = [nm.ProbitRaschNode(), nm.ProbitThreePLNode()]
fit = ft.irtree_fit(data, spec_dis, node_models, n_iter=N_ITER, n_chains=N_CHAINS, seed=SEED+11)
compare(a2_true, 'a_2_', fit, 'node-2 discrimination a (probit)')
compare(c2_true, 'c_2_', fit, 'node-2 guessing c (probit)')

**Logit** — the same study, logit link

In [ ]:
data = ing.load_data(resp_logit, spec=spec_dis, categories_start=1)
node_models = [nm.LogitRaschNode(), nm.LogitThreePLNode()]    # a, c both estimated by MH
fit = ft.irtree_fit(data, spec_dis, node_models, n_iter=N_ITER, n_chains=N_CHAINS, seed=SEED+11)
acc = fit.results[0].accept_rates
print('a MH accept:', {k:round(v,3) for k,v in acc.items() if k.startswith('a_mh')})
print('c MH accept:', {k:round(v,3) for k,v in acc.items() if k.startswith('c_mh')})
compare(a2_true, 'a_2_', fit, 'node-2 discrimination a (logit, MH)')
compare(c2_true, 'c_2_', fit, 'node-2 guessing c (logit, MH)')

## B3. Inspecting the MH machinery on a logit node

Every logit cell above already exercises MH; this one looks directly at the acceptance rates and the evidence-mode freeze reporting.

**Simulate & analyze** (logit node 2)

In [ ]:
spec_dis = ts.disengagement_tree(binary=True)
rng = np.random.default_rng(SEED+12)
theta1, theta2 = rng.normal(0,1,P), rng.normal(0,1,P)
b1, b2 = rng.normal(0.6,0.7,I), rng.normal(0,1,I)
truth = [dict(theta=theta1, b=b1), dict(theta=theta2, b=b2)]
Y = h.simulate_irtree(spec_dis, truth, P, I, np.random.default_rng(SEED+12), node_links=['probit','logit'])
data = ing.load_data(responses_frame(Y), spec=spec_dis, categories_start=1)
node_models = [nm.ProbitRaschNode(), nm.LogitRaschNode()]   # node 2 -> person-wise theta and item-wise b MH blocks
fit = ft.irtree_fit(data, spec_dis, node_models, n_iter=N_ITER, n_chains=N_CHAINS, seed=SEED+12)
res = fit.results[0]
print('MH acceptance (theta_mh_<dim>, b_mh_<group>):',
      {k:round(v,3) for k,v in res.accept_rates.items() if k.startswith(('theta_mh', 'b_mh'))})
print('evidence warmup froze at:', res.frozen_at)
print('effective warmup / n_iter:', res.effective_warmup, '/', res.n_iter)
compare(b2, 'b_2_', fit, 'node-2 difficulty (logit MH node)')

## B4. Disengagement + person covariate (β) on node 1

**Simulate** (data-generating step — separate from the package)

In [ ]:
spec_dis = ts.disengagement_tree(binary=True)
rng = np.random.default_rng(SEED+13)
Xmat = rng.normal(size=(P,2)); Xmat -= Xmat.mean(0)          # centre the covariates
beta_true = np.array([0.5, -0.4])
theta1 = rng.normal(0,1,P) + Xmat @ beta_true               # covariate drives node-1 trait
theta2 = rng.normal(0,1,P)
b1, b2 = rng.normal(0.6,0.7,I), rng.normal(0,1,I)
truth = [dict(theta=theta1, b=b1), dict(theta=theta2, b=b2)]
Y_probit = h.simulate_irtree(spec_dis, truth, P, I, np.random.default_rng(SEED+13), node_links=['probit','probit'])
Y_logit  = h.simulate_irtree(spec_dis, truth, P, I, np.random.default_rng(SEED+13), node_links=['logit','logit'])
resp_probit, resp_logit = responses_frame(Y_probit), responses_frame(Y_logit)
xdf = person_frame(Xmat, ['escs','female'])                 # the person covariate table

**Probit** — ingest, node models, fit, compare

In [ ]:
data = ing.load_data(resp_probit, spec=spec_dis, person_covariates=xdf, categories_start=1)
node_models = [nm.ProbitRaschNode(person_covariates=['escs','female']), nm.ProbitRaschNode()]
fit = ft.irtree_fit(data, spec_dis, node_models, n_iter=N_ITER, n_chains=N_CHAINS, seed=SEED+13)
compare(beta_true, 'beta_', fit, 'person-covariate effects beta (probit)')

**Logit** — the same study, logit link

In [ ]:
data = ing.load_data(resp_logit, spec=spec_dis, person_covariates=xdf, categories_start=1)
node_models = [nm.LogitRaschNode(person_covariates=['escs','female']), nm.LogitRaschNode()]
fit = ft.irtree_fit(data, spec_dis, node_models, n_iter=N_ITER, n_chains=N_CHAINS, seed=SEED+13)
compare(beta_true, 'beta_', fit, 'person-covariate effects beta (logit; covariate through MH offset)')

## B5. Two-node tree + estimated θ-correlation (LKJ)

**Simulate** (data-generating step — separate from the package)

In [ ]:
spec_dis = ts.disengagement_tree(binary=True)
rng = np.random.default_rng(SEED+14)
rho_true = -0.5
L  = np.linalg.cholesky(np.array([[1, rho_true],[rho_true, 1]]))
TH = rng.standard_normal((P,2)) @ L.T
b1, b2 = rng.normal(0.6,0.7,I), rng.normal(0,1,I)
truth = [dict(theta=TH[:,0], b=b1), dict(theta=TH[:,1], b=b2)]
Y_probit = h.simulate_irtree(spec_dis, truth, P, I, np.random.default_rng(SEED+14), node_links=['probit','probit'])
Y_logit  = h.simulate_irtree(spec_dis, truth, P, I, np.random.default_rng(SEED+14), node_links=['logit','logit'])
resp_probit, resp_logit = responses_frame(Y_probit), responses_frame(Y_logit)
print('true latent correlation rho =', rho_true)

**Probit** — ingest, node models, fit, compare

In [ ]:
data = ing.load_data(resp_probit, spec=spec_dis, categories_start=1)
node_models = [nm.ProbitRaschNode(), nm.ProbitRaschNode()]
priors = pr.build_default_priors(spec_dis, node_models)
priors.set('theta.correlation', pr.LKJPrior(eta=2.0))        # freely estimate rho
fit = ft.irtree_fit(data, spec_dis, node_models, priors=priors, n_iter=N_ITER, n_chains=N_CHAINS, seed=SEED+14)
rho_names = [n for n in fit.results[0].param_names if 'rho' in n or ('corr' in n and 'matrix' not in n)]
print('rho parameter(s):', rho_names)
if rho_names: compare([rho_true], rho_names[0], fit, 'latent correlation (probit)')

**Logit** — the same study, logit link

In [ ]:
data = ing.load_data(resp_logit, spec=spec_dis, categories_start=1)
node_models = [nm.LogitRaschNode(), nm.LogitRaschNode()]
priors = pr.build_default_priors(spec_dis, node_models)
priors.set('theta.correlation', pr.LKJPrior(eta=2.0))
fit = ft.irtree_fit(data, spec_dis, node_models, priors=priors, n_iter=N_ITER, n_chains=N_CHAINS, seed=SEED+14)
rho_names = [n for n in fit.results[0].param_names if 'rho' in n or ('corr' in n and 'matrix' not in n)]
if rho_names: compare([rho_true], rho_names[0], fit, 'latent correlation (logit)')